# Modelo reproducible de análisis de cooperación educativa — APC Colombia

**Objetivo del notebook:** reproducir desde la base original de APC-Colombia la aplicación metodológica y financiera de la investigación sobre cooperación internacional asociada al ODS 4.

El flujo reproduce y audita:

**43.105 registros → 36.864 registros 2015–2024 → 1.186 filas ODS 4 → 274 intervenciones → 269 intervenciones con monto financiero confirmado.**

También calcula:

- suma bruta de las filas ODS 4;
- aportes internacionales únicos;
- valor financiero confirmado;
- Gini sobre las 269 intervenciones con monto confirmado;
- HHI, Top 3 y Top 5 por grupo de donante;
- concentración de proyectos ≥ USD 500.000;
- serie por año de registro;
- pruebas de estrés;
- preselección reproducible del núcleo de educación inclusiva;
- tablas de auditoría y exportación de resultados.

> **Principio metodológico:** cada `CODIGO INTERVENCION` distinto se conserva como una intervención independiente. No se fusionan códigos distintos aunque compartan nombre, actor o monto.

> **Principio financiero:** los cinco proyectos sin monto atribuible a un actor con rol explícito de `Donante` permanecen dentro de las 274 intervenciones, pero no se interpretan como proyectos de valor cero para el Gini.

In [ ]:
# 2. Librerías y configuración general
import os
import re
import json
import math
import shutil
import zipfile
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)

START_YEAR = 2015
END_YEAR = 2024
HIGH_VALUE_THRESHOLD = 500_000

OUTPUT_DIR = Path("resultados_tesis_apc")
FIG_DIR = OUTPUT_DIR / "graficas"
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Configuración lista.")

## 3. Cargar la base original de APC
Se utiliza `Data Tesis.xlsx` del directorio de trabajo. Si no existe (por ejemplo en Colab), se descarga de la rama `main` del repositorio público. No es necesario cargar archivos manualmente. Se requiere la hoja `Data`. Para conservar una versión exacta, use el mismo commit del código y de la base.


In [ ]:
from urllib.request import urlopen
SOURCE_FILE = Path("Data Tesis.xlsx")
DATA_URL = "https://raw.githubusercontent.com/Leidyhc12/TFG_Master/main/Data%20Tesis.xlsx"
if not SOURCE_FILE.exists():
    try:
        with urlopen(DATA_URL, timeout=90) as response:
            data_bytes = response.read()
        SOURCE_FILE.write_bytes(data_bytes)
    except Exception as exc:
        raise RuntimeError("No se pudo descargar la base. Verifica la conexión y que el repositorio sea público; alternativamente coloca Data Tesis.xlsx junto al notebook.") from exc
print("Archivo seleccionado:", SOURCE_FILE)

df_raw = pd.read_excel(SOURCE_FILE, sheet_name="Data")

# Normalizar espacios en nombres de columnas para evitar diferencias como
# 'NOMBRE  DEL ACTOR' vs. 'NOMBRE DEL ACTOR'.
df_raw.columns = [
    re.sub(r"\s+", " ", str(c)).strip()
    for c in df_raw.columns
]

print("Filas fuente:", len(df_raw))
print("Columnas:", len(df_raw.columns))
display(df_raw.head(3))

## 4. Funciones auxiliares y validación de columnas

Las funciones de normalización se utilizan para comparar texto sin depender de mayúsculas, tildes o espacios dobles.

In [ ]:
REQUIRED_COLUMNS = [
    "CODIGO INTERVENCION",
    "FECHA REGISTRO INTERVENCION",
    "NOMBRE INTERVENCION",
    "OBJETIVO GENERAL",
    "FECHA INICIAL",
    "FECHA FINAL",
    "ESTADO INTERVENCION",
    "MODALIDAD COOPERACION",
    "ODS",
    "SECTORES GOBIERNO",
    "NOMBRE DEL ACTOR",
    "TIPO DE ACTOR",
    "PAIS DEL ACTOR",
    "ROL DEL ACTOR",
    "TIPO DE ASISTENCIA",
    "MONTO APORTE EN USD",
    "ES ADICION",
]

missing = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
if missing:
    raise KeyError(f"Faltan columnas requeridas: {missing}")

def norm_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip()).upper()

def strip_accents(value):
    if pd.isna(value):
        return ""
    txt = unicodedata.normalize("NFKD", str(value))
    return "".join(ch for ch in txt if not unicodedata.combining(ch)).lower()

def unique_join(series):
    out = []
    seen = set()
    for value in series:
        if pd.isna(value):
            continue
        value = str(value).strip()
        if value and value not in seen:
            out.append(value)
            seen.add(value)
    return " | ".join(out)

def canonical_donor(actor):
    a = strip_accents(actor).strip()

    if "global affairs canada" in a or a == "canada":
        return "Canadá"
    if "abu dhabi" in a:
        return "Emiratos Árabes Unidos"
    if "noruega" in a:
        return "Noruega"
    if any(x in a for x in [
        "estados unidos para el desarrollo internacional",
        "departamento de estado de los estados unidos",
        "departamento de estado de ee.uu",
        "bureau of population, refugees and migration",
        "asuntos internacionales de narcoticos",
    ]):
        return "Estados Unidos - Gobierno"
    if "howard g. buffet" in a or "howard g buffett" in a:
        return "Howard G. Buffett Foundation"
    if a == "corea":
        return "Corea del Sur"
    if "banco interamericano de desarrollo" in a:
        return "BID"
    if "japon" in a:
        return "Japón"
    if "fondo de las naciones unidas para la infancia" in a or a == "unicef":
        return "UNICEF"
    if "union europea" in a:
        return "Unión Europea"
    if "agencia espanola" in a:
        return "España - AECID"
    if "fondo naciones unidas" in a:
        return "Naciones Unidas - Fondo"
    if "acnur" in a:
        return "ACNUR"
    if "coordinacion de asuntos humanitarios" in a or "ocha" in a:
        return "OCHA"
    if "central emergency response fund" in a or "cerf" in a:
        return "CERF"
    if "banco de desarrollo de america latina" in a:
        return "CAF"
    if "reino unido" in a:
        return "Reino Unido"
    if "hilton foundation" in a:
        return "Hilton Foundation"
    if "suecia" in a:
        return "Suecia"
    if "suiza" in a:
        return "Suiza"
    if "alemania" in a or "giz" in a:
        return "Alemania"
    if "belgica" in a:
        return "Bélgica"

    return str(actor).strip()

print("Columnas validadas.")

## 5. Delimitación temporal y filtro ODS 4

La investigación utiliza la **fecha de registro de la intervención** como criterio temporal.

Por tanto, los proyectos registrados antes de 2015 quedan fuera del universo principal, aunque su ejecución haya continuado después.

In [ ]:
df = df_raw.copy()

df["FECHA REGISTRO INTERVENCION"] = pd.to_datetime(
    df["FECHA REGISTRO INTERVENCION"], errors="coerce"
)
df["ANIO_REGISTRO"] = df["FECHA REGISTRO INTERVENCION"].dt.year

df_periodo = df[
    df["ANIO_REGISTRO"].between(START_YEAR, END_YEAR, inclusive="both")
].copy()

ods_text = df_periodo["ODS"].fillna("").astype(str).str.strip()
df_ods4 = df_periodo[
    ods_text.str.match(r"^(?:4\.|04\.)", na=False)
].copy()

n_codigos = df_ods4["CODIGO INTERVENCION"].astype(str).str.strip().nunique()
suma_bruta = pd.to_numeric(
    df_ods4["MONTO APORTE EN USD"], errors="coerce"
).fillna(0).sum()

embudo = pd.DataFrame({
    "Etapa": [
        "Data original",
        "2015–2024",
        "Registros ODS 4",
        "Intervenciones ODS 4",
    ],
    "Cantidad": [
        len(df_raw),
        len(df_periodo),
        len(df_ods4),
        n_codigos,
    ],
})

display(embudo)
print(f"Suma bruta ODS 4: USD {suma_bruta:,.2f}")

## 6. Construcción de aportes financieros únicos

La base APC es relacional: una intervención puede aparecer en varias filas por actor, sector, ODS u otras relaciones.

Por eso **no se suma directamente cada fila como si fuera un desembolso independiente**.

Se conserva una contribución una sola vez utilizando la llave:

`CODIGO INTERVENCION + ACTOR + ROL + TIPO DE ASISTENCIA + MONTO + ES ADICION`

Además, para el análisis por fuente se exige:

- `MONTO APORTE EN USD > 0`
- `ROL DEL ACTOR` contiene `Donante`
- `TIPO DE ACTOR` contiene `INTERNACIONAL`

In [ ]:
df_ods4["MONTO_APORTE_USD_NUM"] = pd.to_numeric(
    df_ods4["MONTO APORTE EN USD"], errors="coerce"
).fillna(0.0)

df_ods4["ACTOR_NORM"] = df_ods4["NOMBRE DEL ACTOR"].map(norm_text)
df_ods4["ROL_NORM"] = df_ods4["ROL DEL ACTOR"].map(norm_text)
df_ods4["ASISTENCIA_NORM"] = df_ods4["TIPO DE ASISTENCIA"].map(norm_text)
df_ods4["ADICION_NORM"] = df_ods4["ES ADICION"].map(norm_text)
df_ods4["TIPO_ACTOR_NORM"] = df_ods4["TIPO DE ACTOR"].map(norm_text)
df_ods4["MONTO_ROUND"] = df_ods4["MONTO_APORTE_USD_NUM"].round(6)

mask_donante = (
    (df_ods4["MONTO_APORTE_USD_NUM"] > 0)
    & df_ods4["ROL_NORM"].str.contains("DONANTE", na=False)
)

aporte_key = [
    "CODIGO INTERVENCION",
    "ACTOR_NORM",
    "ROL_NORM",
    "ASISTENCIA_NORM",
    "MONTO_ROUND",
    "ADICION_NORM",
]

df_aportes_unicos = (
    df_ods4.loc[mask_donante]
    .drop_duplicates(subset=aporte_key)
    .copy()
)

df_aportes_unicos["ACTOR_INTERNACIONAL"] = (
    df_aportes_unicos["TIPO_ACTOR_NORM"]
    .str.contains("INTERNACIONAL", na=False)
)

df_aportes_int = df_aportes_unicos[
    df_aportes_unicos["ACTOR_INTERNACIONAL"]
].copy()

df_aportes_int["DONANTE_GRUPO"] = (
    df_aportes_int["NOMBRE DEL ACTOR"].map(canonical_donor)
)

valor_confirmado = df_aportes_int["MONTO_APORTE_USD_NUM"].sum()

print("Aportes únicos con donante explícito:", len(df_aportes_unicos))
print("Aportes internacionales únicos:", len(df_aportes_int))
print(f"Valor financiero confirmado: USD {valor_confirmado:,.2f}")

## 7. Construcción del universo de 274 intervenciones

Aquí **cada código distinto permanece como una intervención independiente**.

El monto financiero de cada intervención se obtiene sumando únicamente sus aportes internacionales únicos con fuente donante explícita.

In [ ]:
donor_by_code = (
    df_aportes_int
    .groupby("CODIGO INTERVENCION")["DONANTE_GRUPO"]
    .agg(lambda s: " | ".join(sorted(set(map(str, s)))))
    .rename("DONANTES_NORMALIZADOS")
)

amount_by_code = (
    df_aportes_int
    .groupby("CODIGO INTERVENCION")["MONTO_APORTE_USD_NUM"]
    .sum()
    .rename("MONTO_USD_CONFIRMADO")
)

n_aportes_by_code = (
    df_aportes_int
    .groupby("CODIGO INTERVENCION")
    .size()
    .rename("N_APORTES_EXTERNOS")
)

project_base = (
    df_ods4
    .sort_values(["CODIGO INTERVENCION", "FECHA REGISTRO INTERVENCION"])
    .groupby("CODIGO INTERVENCION", as_index=False)
    .agg({
        "ANIO_REGISTRO": "first",
        "FECHA REGISTRO INTERVENCION": "first",
        "FECHA INICIAL": "first",
        "FECHA FINAL": "first",
        "NOMBRE INTERVENCION": "first",
        "OBJETIVO GENERAL": "first",
        "ESTADO INTERVENCION": "first",
        "MODALIDAD COOPERACION": "first",
        "ODS": unique_join,
        "SECTORES GOBIERNO": unique_join,
    })
)

df_proyectos = (
    project_base
    .merge(amount_by_code, on="CODIGO INTERVENCION", how="left")
    .merge(n_aportes_by_code, on="CODIGO INTERVENCION", how="left")
    .merge(donor_by_code, on="CODIGO INTERVENCION", how="left")
)

df_proyectos["MONTO_USD_CONFIRMADO"] = (
    df_proyectos["MONTO_USD_CONFIRMADO"].fillna(0.0)
)
df_proyectos["N_APORTES_EXTERNOS"] = (
    df_proyectos["N_APORTES_EXTERNOS"].fillna(0).astype(int)
)
df_proyectos["DONANTES_NORMALIZADOS"] = (
    df_proyectos["DONANTES_NORMALIZADOS"].fillna("")
)

df_proyectos["TIENE_MONTO_CONFIRMADO"] = np.where(
    df_proyectos["MONTO_USD_CONFIRMADO"] > 0, "SI", "NO"
)
df_proyectos["ALTO_VALOR_500K"] = np.where(
    df_proyectos["MONTO_USD_CONFIRMADO"] >= HIGH_VALUE_THRESHOLD, "SI", "NO"
)

df_financiero = df_proyectos[
    df_proyectos["MONTO_USD_CONFIRMADO"] > 0
].copy()

print("Universo temático:", len(df_proyectos))
print("Universo financiero:", len(df_financiero))
print("Sin monto financiero confirmado:", len(df_proyectos) - len(df_financiero))

## 8. Auditoría de los cinco proyectos sin monto financiero confirmado

Estos proyectos **no son proyectos de valor cero**.

La interpretación correcta es: la base original contiene montos positivos, pero bajo roles que no permiten atribuirlos inequívocamente a un `Donante` utilizando el criterio financiero adoptado.

In [ ]:
codes_sin_monto = set(
    df_proyectos.loc[
        df_proyectos["MONTO_USD_CONFIRMADO"].eq(0),
        "CODIGO INTERVENCION"
    ].astype(str)
)

original_positive_no_confirmed = df_ods4[
    df_ods4["CODIGO INTERVENCION"].astype(str).isin(codes_sin_monto)
    & (df_ods4["MONTO_APORTE_USD_NUM"] > 0)
].copy()

df_control_5 = (
    original_positive_no_confirmed
    .groupby("CODIGO INTERVENCION", as_index=False)
    .agg({
        "NOMBRE INTERVENCION": "first",
        "NOMBRE DEL ACTOR": unique_join,
        "ROL DEL ACTOR": unique_join,
        "MONTO_APORTE_USD_NUM": lambda s: " | ".join(
            f"{x:,.2f}" for x in sorted(set(map(float, s)))
        ),
    })
    .rename(columns={
        "NOMBRE DEL ACTOR": "ACTORES_CON_MONTO",
        "ROL DEL ACTOR": "ROLES_REGISTRADOS",
        "MONTO_APORTE_USD_NUM": "MONTOS_OBSERVADOS_USD",
    })
)

display(df_control_5)

## 9. Indicadores financieros

### Gini
Se calcula exclusivamente sobre las **269 intervenciones con monto financiero confirmado > 0**.

Esto evita interpretar la ausencia de un monto atribuible a `Donante` como si el proyecto realmente hubiese tenido USD 0.

### HHI y concentración por donante
Se calculan sobre los aportes internacionales únicos con fuente explícita.

In [ ]:
def gini(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]

    if len(x) == 0 or x.sum() == 0:
        return np.nan

    x = np.sort(x)
    n = len(x)
    return (
        2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())
        - (n + 1) / n
    )

gini_financiero = gini(df_financiero["MONTO_USD_CONFIRMADO"])

df_donantes = (
    df_aportes_int
    .groupby("DONANTE_GRUPO", as_index=False)["MONTO_APORTE_USD_NUM"]
    .sum()
    .rename(columns={"MONTO_APORTE_USD_NUM": "VALOR_USD"})
    .sort_values("VALOR_USD", ascending=False)
    .reset_index(drop=True)
)

df_donantes["PARTICIPACION"] = df_donantes["VALOR_USD"] / df_donantes["VALOR_USD"].sum()
df_donantes["COMPONENTE_HHI"] = df_donantes["PARTICIPACION"] ** 2

hhi = df_donantes["COMPONENTE_HHI"].sum()
top3 = df_donantes.head(3)["PARTICIPACION"].sum()
top5 = df_donantes.head(5)["PARTICIPACION"].sum()

df_high = df_financiero[
    df_financiero["MONTO_USD_CONFIRMADO"] >= HIGH_VALUE_THRESHOLD
].copy()

high_share = (
    df_high["MONTO_USD_CONFIRMADO"].sum()
    / df_financiero["MONTO_USD_CONFIRMADO"].sum()
)

resumen_financiero = pd.DataFrame({
    "Indicador": [
        "Intervenciones ODS 4",
        "Intervenciones con monto confirmado",
        "Intervenciones sin monto confirmado",
        "Suma bruta ODS 4 (USD)",
        "Valor financiero confirmado (USD)",
        "Valor confirmado / suma bruta",
        "Proyectos >= USD 500.000",
        "Participación proyectos >= USD 500.000",
        "Gini financiero",
        "HHI donantes",
        "Top 3 donantes",
        "Top 5 donantes",
    ],
    "Resultado": [
        len(df_proyectos),
        len(df_financiero),
        len(df_proyectos) - len(df_financiero),
        suma_bruta,
        valor_confirmado,
        valor_confirmado / suma_bruta,
        len(df_high),
        high_share,
        gini_financiero,
        hhi,
        top3,
        top5,
    ]
})

display(resumen_financiero)

## 10. Verificación de reproducibilidad

Esta celda compara los resultados obtenidos con la versión de la base APC utilizada en la investigación.

Si en el futuro APC actualiza el archivo, algún resultado puede cambiar. En ese caso, la celda mostrará `REVISAR` en lugar de ocultar la diferencia.

In [ ]:
EXPECTED = {
    "registros_raw": 43105,
    "registros_periodo": 36864,
    "filas_ods4": 1186,
    "intervenciones_ods4": 274,
    "intervenciones_financieras": 269,
    "aportes_internacionales_unicos": 354,
    "suma_bruta": 250_431_246.8416312,
    "valor_confirmado": 111_251_610.669956,
    "gini": 0.7784110748282991,
    "hhi": 0.15119039651673985,
    "top3": 0.5823722563607502,
    "top5": 0.725465397290494,
    "high_n": 30,
    "high_share": 0.7544995426687326,
}

ACTUAL = {
    "registros_raw": len(df_raw),
    "registros_periodo": len(df_periodo),
    "filas_ods4": len(df_ods4),
    "intervenciones_ods4": len(df_proyectos),
    "intervenciones_financieras": len(df_financiero),
    "aportes_internacionales_unicos": len(df_aportes_int),
    "suma_bruta": suma_bruta,
    "valor_confirmado": valor_confirmado,
    "gini": gini_financiero,
    "hhi": hhi,
    "top3": top3,
    "top5": top5,
    "high_n": len(df_high),
    "high_share": high_share,
}

rows_check = []
for key, expected in EXPECTED.items():
    actual = ACTUAL[key]

    if isinstance(expected, (float, np.floating)):
        ok = np.isclose(actual, expected, rtol=1e-9, atol=1e-6)
    else:
        ok = actual == expected

    rows_check.append({
        "Indicador": key,
        "Esperado": expected,
        "Obtenido": actual,
        "Estado": "OK" if ok else "REVISAR",
    })

df_check = pd.DataFrame(rows_check)
display(df_check)

if (df_check["Estado"] == "OK").all():
    print("✅ Los resultados principales fueron reproducidos correctamente.")
else:
    print("⚠️ Hay diferencias. Verifica si la versión del archivo APC cambió.")

## 11. Serie por año de registro

**Importante:** esta serie representa el valor de aportes/proyectos **registrados** en cada año.  
No debe denominarse desembolso anual ni flujo anual de caja.

In [ ]:
df_serie = (
    df_aportes_int
    .groupby("ANIO_REGISTRO", as_index=False)["MONTO_APORTE_USD_NUM"]
    .sum()
    .rename(columns={"MONTO_APORTE_USD_NUM": "VALOR_USD"})
)

df_serie = (
    pd.DataFrame({"ANIO_REGISTRO": range(START_YEAR, END_YEAR + 1)})
    .merge(df_serie, on="ANIO_REGISTRO", how="left")
    .fillna({"VALOR_USD": 0})
)

df_serie["VARIACION_INTERANUAL"] = df_serie["VALOR_USD"].pct_change()
df_serie["INDICE_2015_100"] = (
    df_serie["VALOR_USD"]
    / df_serie.loc[df_serie["ANIO_REGISTRO"].eq(START_YEAR), "VALOR_USD"].iloc[0]
    * 100
)

display(df_serie)

## 12. Curva de Lorenz y Gini

In [ ]:
values = np.sort(df_financiero["MONTO_USD_CONFIRMADO"].to_numpy(dtype=float))
cum_values = np.insert(np.cumsum(values) / values.sum(), 0, 0)
cum_projects = np.linspace(0, 1, len(values) + 1)

plt.figure(figsize=(7, 6))
plt.plot(cum_projects, cum_values, label=f"Lorenz (Gini={gini_financiero:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Igualdad perfecta")
plt.xlabel("Proporción acumulada de intervenciones")
plt.ylabel("Proporción acumulada del valor")
plt.title("Curva de Lorenz — universo financiero")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "01_lorenz_gini.png", dpi=180)
plt.show()

## 13. Principales donantes y concentración

In [ ]:
display(df_donantes.head(15))

top10 = df_donantes.head(10).sort_values("VALOR_USD")

plt.figure(figsize=(9, 6))
plt.barh(top10["DONANTE_GRUPO"], top10["VALOR_USD"])
plt.xlabel("Valor USD")
plt.title("Top 10 grupos de donantes — ODS 4")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_top10_donantes.png", dpi=180)
plt.show()

## 14. Concentración de proyectos de alto valor

In [ ]:
df_ranking = (
    df_financiero
    .sort_values("MONTO_USD_CONFIRMADO", ascending=False)
    .copy()
)

df_ranking["PARTICIPACION"] = (
    df_ranking["MONTO_USD_CONFIRMADO"]
    / df_ranking["MONTO_USD_CONFIRMADO"].sum()
)
df_ranking["PARTICIPACION_ACUMULADA"] = df_ranking["PARTICIPACION"].cumsum()

print("Proyectos >= USD 500.000:", len(df_high))
print(f"Participación sobre valor confirmado: {high_share:.2%}")

display(
    df_ranking[
        ["CODIGO INTERVENCION", "NOMBRE INTERVENCION", "MONTO_USD_CONFIRMADO",
         "PARTICIPACION", "PARTICIPACION_ACUMULADA"]
    ].head(20)
)

## 15. Pruebas de estrés

Las pruebas de estrés **no son pronósticos 2026–2030**.  
Miden exposición financiera hipotética frente a reducciones en los tres principales grupos de donantes.

In [ ]:
top3_usd = df_donantes.head(3)["VALOR_USD"].sum()

shocks = [0.10, 0.25, 0.50, 0.75, 1.00]

df_stress = pd.DataFrame({"SHOCK_TOP3": shocks})
df_stress["PERDIDA_POTENCIAL_USD"] = df_stress["SHOCK_TOP3"] * top3_usd
df_stress["PERDIDA_PCT_PORTAFOLIO"] = (
    df_stress["PERDIDA_POTENCIAL_USD"] / valor_confirmado
)
df_stress["PORTAFOLIO_REMANENTE_USD"] = (
    valor_confirmado - df_stress["PERDIDA_POTENCIAL_USD"]
)
df_stress["PORTAFOLIO_REMANENTE_PCT"] = (
    df_stress["PORTAFOLIO_REMANENTE_USD"] / valor_confirmado
)

display(df_stress)

plt.figure(figsize=(8, 5))
plt.bar(
    [f"{int(s*100)}%" for s in df_stress["SHOCK_TOP3"]],
    df_stress["PERDIDA_POTENCIAL_USD"]
)
plt.xlabel("Reducción hipotética del Top 3")
plt.ylabel("Pérdida potencial USD")
plt.title("Prueba de estrés — exposición al Top 3 de donantes")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_stress_top3.png", dpi=180)
plt.show()

# 16. Preselección del núcleo de educación inclusiva

Este módulo es **semiautomático**.

Python identifica intervenciones con señales textuales relacionadas con educación y poblaciones vulnerables.  
El resultado es una **preselección**, no una clasificación definitiva.

La validación humana sigue siendo necesaria para evitar falsos positivos, por ejemplo cuando términos como `género` o `discapacidad` aparecen solamente en la descripción estándar de una meta ODS.

In [ ]:
EDU_TERMS = [
    "educa", "aprendiz", "escuel", "escolar", "formacion", "capacit",
    "primera infancia", "desarrollo infantil", "beca", "universidad",
    "universitari", "docent", "ensenanza", "curricul", "competenc",
    "habilidad", "alfabet", "pedagog", "colegio", "estudiante",
]

VULNERABILITY_TERMS = [
    "migran", "refug", "discap", "rural", "victim", "etnic", "indigen",
    "afro", "poblacion vulnerable", "vulnerab", "exclusion", "inclusion",
    "equidad", "desigual", "frontera", "conflicto armado", "desplaz",
    "reintegr", "mujer", "genero", "lgbti", "pobreza",
]

DIRECT_POPULATION_TERMS = [
    "migran", "refug", "discap", "victim", "indigen", "afro", "rural", "desplaz",
]

def classify_inclusion(row):
    text = strip_accents(" ".join([
        str(row.get("NOMBRE INTERVENCION", "")),
        str(row.get("OBJETIVO GENERAL", "")),
        str(row.get("ODS", "")),
        str(row.get("SECTORES GOBIERNO", "")),
    ]))

    hits = sorted({
        term for term in VULNERABILITY_TERMS
        if term in text
    })

    if "educacion inclusiva" in text or "inclusion educativa" in text:
        classification = "A - Inclusión explícita"
    elif (
        any(term in text for term in DIRECT_POPULATION_TERMS)
        and any(term in text for term in EDU_TERMS)
    ):
        classification = "A - Población vulnerable + educación"
    elif hits:
        classification = "B - Potencialmente inclusivo"
    else:
        classification = "C - Educación general"

    return pd.Series({
        "CLASIFICACION_PRELIMINAR": classification,
        "PALABRAS_CLAVE_VULNERABILIDAD": ", ".join(hits),
    })

classification = df_proyectos.apply(classify_inclusion, axis=1)
df_proyectos = pd.concat([df_proyectos, classification], axis=1)

df_nucleo_preliminar = df_proyectos[
    df_proyectos["CLASIFICACION_PRELIMINAR"].str.startswith("A")
].copy()

df_nucleo_preliminar["VALIDACION_MANUAL"] = ""
df_nucleo_preliminar["JUSTIFICACION_MANUAL"] = ""

print("Núcleo preliminar:", len(df_nucleo_preliminar))
print(
    "Núcleo preliminar con monto confirmado:",
    (df_nucleo_preliminar["MONTO_USD_CONFIRMADO"] > 0).sum()
)

display(
    df_nucleo_preliminar[
        [
            "CODIGO INTERVENCION",
            "NOMBRE INTERVENCION",
            "OBJETIVO GENERAL",
            "CLASIFICACION_PRELIMINAR",
            "PALABRAS_CLAVE_VULNERABILIDAD",
            "MONTO_USD_CONFIRMADO",
            "VALIDACION_MANUAL",
            "JUSTIFICACION_MANUAL",
        ]
    ].head(20)
)

## 17. Cómo validar manualmente el núcleo

No edites la lógica financiera.

La revisión manual solo responde esta pregunta:

> **¿La intervención está dirigida de manera explícita a reducir barreras educativas, desigualdades o atender una población vulnerable?**

Marca:

- `SI` → inclusión directa suficientemente sustentada;
- `NO` → educación general o falso positivo;
- `REVISAR` → evidencia insuficiente.

La columna `JUSTIFICACION_MANUAL` debe contener una frase corta basada en el nombre/objetivo del proyecto.

Puedes exportar esta tabla, revisarla en Google Sheets y volver a cargarla posteriormente.

In [ ]:
review_cols = [
    "CODIGO INTERVENCION",
    "NOMBRE INTERVENCION",
    "OBJETIVO GENERAL",
    "ODS",
    "SECTORES GOBIERNO",
    "CLASIFICACION_PRELIMINAR",
    "PALABRAS_CLAVE_VULNERABILIDAD",
    "MONTO_USD_CONFIRMADO",
    "VALIDACION_MANUAL",
    "JUSTIFICACION_MANUAL",
]

review_path = OUTPUT_DIR / "Revision_Manual_Nucleo_Inclusivo.csv"
df_nucleo_preliminar[review_cols].to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo generado:", review_path)

## 18. Serie gráfica por año de registro

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(df_serie["ANIO_REGISTRO"], df_serie["VALOR_USD"], marker="o")
plt.xlabel("Año de registro")
plt.ylabel("Valor confirmado USD")
plt.title("Valor financiero confirmado por año de registro")
plt.xticks(df_serie["ANIO_REGISTRO"])
plt.tight_layout()
plt.savefig(FIG_DIR / "04_serie_anual.png", dpi=180)
plt.show()

# 19. Exportar resultados

Se generan:

- un Excel con tablas de auditoría y resultados;
- CSV del núcleo para validación manual;
- gráficos PNG;
- JSON con los principales indicadores;
- ZIP final para descarga.

El Excel es un producto del modelo; **el cálculo se realiza en Python**.

In [ ]:
# Preparar tablas para exportación
df_aportes_export = df_aportes_int[
    [
        "CODIGO INTERVENCION",
        "ANIO_REGISTRO",
        "FECHA REGISTRO INTERVENCION",
        "NOMBRE INTERVENCION",
        "NOMBRE DEL ACTOR",
        "DONANTE_GRUPO",
        "TIPO DE ACTOR",
        "PAIS DEL ACTOR",
        "ROL DEL ACTOR",
        "TIPO DE ASISTENCIA",
        "MONTO_APORTE_USD_NUM",
        "ES ADICION",
    ]
].copy()

summary_json = {
    "registros_raw": int(len(df_raw)),
    "registros_2015_2024": int(len(df_periodo)),
    "filas_ods4": int(len(df_ods4)),
    "intervenciones_ods4": int(len(df_proyectos)),
    "intervenciones_financieras": int(len(df_financiero)),
    "sin_monto_confirmado": int(len(df_proyectos) - len(df_financiero)),
    "suma_bruta_usd": float(suma_bruta),
    "valor_confirmado_usd": float(valor_confirmado),
    "gini": float(gini_financiero),
    "hhi": float(hhi),
    "top3": float(top3),
    "top5": float(top5),
    "proyectos_500k": int(len(df_high)),
    "participacion_500k": float(high_share),
    "nucleo_preliminar": int(len(df_nucleo_preliminar)),
    "nucleo_preliminar_financiero": int(
        (df_nucleo_preliminar["MONTO_USD_CONFIRMADO"] > 0).sum()
    ),
}

with open(OUTPUT_DIR / "Indicadores_Principales.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

excel_path = OUTPUT_DIR / "Resultados_Modelo_Tesis_APC.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    resumen_financiero.to_excel(writer, sheet_name="Resumen", index=False)
    embudo.to_excel(writer, sheet_name="Embudo", index=False)
    df_proyectos.to_excel(writer, sheet_name="Intervenciones_274", index=False)
    df_financiero.to_excel(writer, sheet_name="Universo_Financiero_269", index=False)
    df_control_5.to_excel(writer, sheet_name="Control_5_Sin_Donante", index=False)
    df_aportes_export.to_excel(writer, sheet_name="Aportes_Unicos", index=False)
    df_donantes.to_excel(writer, sheet_name="Donantes", index=False)
    df_serie.to_excel(writer, sheet_name="Serie_Anual", index=False)
    df_ranking.to_excel(writer, sheet_name="Ranking_Proyectos", index=False)
    df_stress.to_excel(writer, sheet_name="Stress_Test", index=False)
    df_nucleo_preliminar.to_excel(writer, sheet_name="Nucleo_Preliminar", index=False)
    df_check.to_excel(writer, sheet_name="Reproducibilidad", index=False)

print("Excel:", excel_path)

## 20. Crear ZIP y descargar

In [ ]:
zip_path = Path("Modelo_Tesis_APC_Resultados.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            z.write(path, arcname=path.relative_to(OUTPUT_DIR))

print("ZIP generado:", zip_path)
if IN_COLAB:
    files.download(str(zip_path))
else:
    print("Resultados disponibles en:", zip_path.resolve())

# 21. Lectura metodológica para la tesis

Una vez reproducidos los resultados, la metodología puede describirse así:

> La aplicación metodológica y financiera se implementó mediante un modelo reproducible desarrollado en Python y ejecutado en Google Colab. El modelo parte de la base original de APC-Colombia y automatiza la delimitación temporal 2015–2024, la identificación de registros asociados al ODS 4, la consolidación de intervenciones por código, la identificación de aportes internacionales únicos, el cálculo del valor financiero confirmado, los indicadores de concentración y las pruebas de estrés. Cada código de intervención distinto se conserva como una unidad independiente. Para los análisis financieros por proyecto se utilizan únicamente las intervenciones con monto positivo atribuible a un actor identificado explícitamente bajo el rol de Donante; las intervenciones sin esta atribución permanecen en el universo temático pero no son tratadas como proyectos de valor cero. La clasificación del núcleo de educación inclusiva se desarrolla mediante una preselección automatizada basada en reglas textuales, complementada con validación humana de los casos seleccionados.

**No se debe afirmar que la serie anual corresponde a desembolsos o flujos de caja.**  
Corresponde al valor de proyectos/aportes registrados en la fuente APC.